In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/suchintikasarkar/sentiment-analysis-for-mental-health/Combined Data.csv


# MindScope — NLP Branch
## DistilBERT Fine-tuning for Depression Detection
**Dataset:** Sentiment Analysis for Mental Health (suchintikasarkar)  
**Model:** distilbert-base-uncased  
**Task:** Binary classification : Depression vs Normal  
**Result:** 0.98 Macro-F1 on test set

In [8]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
x = torch.tensor([1.0]).cuda()
print("GPU works:", x)

2.10.0+cu128
True
Tesla T4
GPU works: tensor([1.], device='cuda:0')


In [9]:
!pip install transformers -q

import pandas as pd
import numpy as np
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import warnings
warnings.filterwarnings('ignore')

print("GPU available:", torch.cuda.is_available())

GPU available: True


In [10]:
df=pd.read_csv('/kaggle/input/datasets/suchintikasarkar/sentiment-analysis-for-mental-health/Combined Data.csv')
print(df.shape)
print(df.columns.tolist())
print("\n Label counts:")
df['status'].value_counts()


(53043, 3)
['Unnamed: 0', 'statement', 'status']

 Label counts:


status
Normal                  16351
Depression              15404
Suicidal                10653
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64

In [11]:
df=df[df['status'].isin(['Depression','Normal'])].copy()
df=df.dropna(subset=['statement','status'])
df=df.reset_index(drop=True)

In [12]:
df['label']=(df['status']=='Depression').astype(int)
print(f"Total samples after filtering:{len(df)}")
print(f"\n label distribution");
print(df['label'].value_counts())
print(f"\nSample statement:")
print(df['statement'].iloc[50])
print(f"Label: {df['status'].iloc[50]}")
print(df.head())

Total samples after filtering:31747

 label distribution
label
0    16343
1    15404
Name: count, dtype: int64

Sample statement:
I really hate bulletsâ€ in pineapple jam
Label: Normal
   Unnamed: 0                                          statement  status  \
0         733      Gr gr dreaming of ex crush to be my game, God  Normal   
1         734                                 wkwkwk what a joke  Normal   
2         735  Leaves are also standby in front of the PC ......  Normal   
3         736     Thank God even though it's just a ride through  Normal   
4         737  wedding teaser concept using the song day6 - o...  Normal   

   label  
0      0  
1      0  
2      0  
3      0  
4      0  


In [13]:
!pip install langdetect -q
from langdetect import detect,LangDetectException

def is_english(text):
    try:
        return detect(text)=='en'
    except LangDetectException:
        return False

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.1 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done


In [14]:
print("Detecting Languages........")
df['is_englist']=df['statement'].apply(is_english)

Detecting Languages........


In [15]:
print(f"\nBefore filtering: {len(df)} rows")
df=df[df['is_englist']].reset_index(drop=True)
print(f"After filtering:{len(df)} rows")
print("\nNew label distribution:")
print(df['label'].value_counts())


Before filtering: 31747 rows
After filtering:29040 rows

New label distribution:
label
1    15152
0    13888
Name: count, dtype: int64


In [18]:
from sklearn.model_selection import train_test_split
train_val_texts,test_texts,train_val_labels,test_labels=train_test_split(df['statement'].tolist(),df['label'].tolist(),test_size=0.2,stratify=df['label'],random_state=42)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts,
    train_val_labels,
    test_size=0.25,
    stratify=train_val_labels,
    random_state=42
)


print(f"Train:       {len(train_texts)} samples")
print(f"Validation:  {len(val_texts)} samples")
print(f"Test:        {len(test_texts)} samples")
print(f"Total:       {len(train_texts)+len(val_texts)+len(test_texts)} samples")

Train:       17424 samples
Validation:  5808 samples
Test:        5808 samples
Total:       29040 samples


In [19]:
from transformers import DistilBertTokenizer
tokenizer=DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
def tokenize(texts):
    return tokenizer(list(texts),padding=True,truncation=True,max_length=256,return_tensors='pt')


print("Tokenizing train set...")
train_encodings = tokenize(train_texts)
print("Tokenizing validation set...")
val_encodings = tokenize(val_texts)
print("Tokenizing test set...")
test_encodings = tokenize(test_texts)
print("Done!")
print(f"\nShape of train input_ids: {train_encodings['input_ids'].shape}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing train set...
Tokenizing validation set...
Tokenizing test set...
Done!

Shape of train input_ids: torch.Size([17424, 256])


In [20]:

from torch.utils.data import Dataset

class DepressionDataset(Dataset):
    def __init__(self,encodings,labels):
        self.encodings=encodings
        self.labels=labels

    def __len__(self):
        return len(self.labels)
    def __getitem__(self,idx):
        item={key:val[idx] for key,val in self.encodings.items()}
        item['labels']=torch.tensor(self.labels[idx])
        return item

train_dataset=DepressionDataset(train_encodings,train_labels)
val_dataset=DepressionDataset(val_encodings,val_labels)
test_dataset=DepressionDataset(test_encodings,test_labels)


print(f"Train dataset size:      {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size:       {len(test_dataset)}")


sample=train_dataset[0]
print(f"\nSample keys:        {list(sample.keys())}")
print(f"input_ids shape:    {sample['input_ids'].shape}")
print(f"Label:              {sample['labels']}")

Train dataset size:      17424
Validation dataset size: 5808
Test dataset size:       5808

Sample keys:        ['input_ids', 'attention_mask', 'labels']
input_ids shape:    torch.Size([256])
Label:              1


In [12]:
from transformers import DistilBertForSequenceClassification
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device{device}")
model=DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased',num_labels=2)
model.to(device)
print("Model loaded and moved to GPU.")
print(f"\nModel parameter: {sum(p.numel() for p in model.parameters()):,}")

Using devicecuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded and moved to GPU.

Model parameter: 66,955,010


In [13]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.metrics import f1_score


train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=32,shuffle=False)
optimizer=AdamW(model.parameters(),lr=2e-5)
best_val_f1=0

for epoch in range(3):
    model.train()
    total_loss=0

    for i,batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids=batch['input_ids'].to(device)
        attention_mask=batch['attention_mask'].to(device)
        labels=batch['labels'].to(device)
        outputs = model(input_ids=input_ids,attention_mask=attention_mask,labels=labels)
        loss=outputs.loss
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
        if(i+1)%100==0:
            
            print(f"Epoch{epoch+1} |Batch {i+1}/{len(train_loader)} | Loss:{loss.item():.4f}")
        
    avg_loss=total_loss/len(train_loader)
    model.eval()
    val_preds,val_true=[],[]
    with torch.no_grad():
        for batch in val_loader:
            input_ids=batch['input_ids'].to(device)
            attention_mask=batch['attention_mask'].to(device)
            labels=batch['labels'].to(device)
            outputs=model(input_ids=input_ids,attention_mask=attention_mask,labels=labels)
            preds=torch.argmax(outputs.logits,dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_true.extend(labels.cpu().numpy())

    val_f1=f1_score(val_true,val_preds,average='macro')
    print(f"\nEpoch:{epoch+1}/3 Complete | Avg Loss:{avg_loss:.4f} | Val Macro-F1:{val_f1:.4f}")
    if val_f1>best_val_f1:
        best_val_f1=val_f1
        model.save_pretrained('/kaggle/working/mindscope_nlp_model')
        tokenizer.save_pretrained('/kaggle/working/mindscope_nlp_model')
        print(f"  → Best model saved (F1: {val_f1:.4f})\n")










Epoch1 |Batch 100/1090 | Loss:0.0994
Epoch1 |Batch 200/1090 | Loss:0.2334
Epoch1 |Batch 300/1090 | Loss:0.0803
Epoch1 |Batch 400/1090 | Loss:0.1131
Epoch1 |Batch 500/1090 | Loss:0.0746
Epoch1 |Batch 600/1090 | Loss:0.2475
Epoch1 |Batch 700/1090 | Loss:0.1237
Epoch1 |Batch 800/1090 | Loss:0.6741
Epoch1 |Batch 900/1090 | Loss:0.0091
Epoch1 |Batch 1000/1090 | Loss:0.0243

Epoch:1/3 Complete | Avg Loss:0.1087 | Val Macro-F1:0.9757


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Best model saved (F1: 0.9757)

Epoch2 |Batch 100/1090 | Loss:0.0021
Epoch2 |Batch 200/1090 | Loss:0.0151
Epoch2 |Batch 300/1090 | Loss:0.0733
Epoch2 |Batch 400/1090 | Loss:0.0012
Epoch2 |Batch 500/1090 | Loss:0.0039
Epoch2 |Batch 600/1090 | Loss:0.0020
Epoch2 |Batch 700/1090 | Loss:0.0024
Epoch2 |Batch 800/1090 | Loss:0.0759
Epoch2 |Batch 900/1090 | Loss:0.0693
Epoch2 |Batch 1000/1090 | Loss:0.0028

Epoch:2/3 Complete | Avg Loss:0.0402 | Val Macro-F1:0.9805


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Best model saved (F1: 0.9805)

Epoch3 |Batch 100/1090 | Loss:0.0368
Epoch3 |Batch 200/1090 | Loss:0.0030
Epoch3 |Batch 300/1090 | Loss:0.1098
Epoch3 |Batch 400/1090 | Loss:0.0019
Epoch3 |Batch 500/1090 | Loss:0.0014
Epoch3 |Batch 600/1090 | Loss:0.0009
Epoch3 |Batch 700/1090 | Loss:0.0004
Epoch3 |Batch 800/1090 | Loss:0.1346
Epoch3 |Batch 900/1090 | Loss:0.0002
Epoch3 |Batch 1000/1090 | Loss:0.0206

Epoch:3/3 Complete | Avg Loss:0.0169 | Val Macro-F1:0.9774


In [21]:
from transformers import DistilBertForSequenceClassification
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device{device}")
best_model=DistilBertForSequenceClassification.from_pretrained('/kaggle/working/mindscope_nlp_model')
best_model.to(device)
best_model.eval()

test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)
test_preds,test_true=[],[]
with torch.no_grad():
    for batch in test_loader:
        input_ids=batch['input_ids'].to(device)
        attention_mask=batch['attention_mask'].to(device)
        labels=batch['labels'].to(device)
        outputs=best_model(input_ids=input_ids,attention_mask=attention_mask)
        preds=torch.argmax(outputs.logits,dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_true.extend(labels.cpu().numpy())


print("=== FINAL TEST SET RESULTS ===")
print(classification_report(test_true, test_preds,
      target_names=['Normal', 'Depression']))


Using devicecuda


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

=== FINAL TEST SET RESULTS ===
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99      2778
  Depression       0.99      0.99      0.99      3030

    accuracy                           0.99      5808
   macro avg       0.99      0.99      0.99      5808
weighted avg       0.99      0.99      0.99      5808



In [24]:
def predict_depression(text):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=256,
        padding=True
    ).to(device)
    
    # Get prediction
    best_model.eval()
    with torch.no_grad():
        outputs = best_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        depression_prob = probs[0][1].item()
        normal_prob = probs[0][0].item()
    
    print(f"Text: {text[:80]}...")
    print(f"Depression probability: {depression_prob:.4f}")
    print(f"Normal probability:     {normal_prob:.4f}")
    print(f"Prediction: {'DEPRESSION' if depression_prob > 0.5 else 'NORMAL'}")
    print("-" * 60)

# Try your own inputs here
# Group 1 — Should clearly be DEPRESSION (high probability)
predict_depression("I haven't left my bed in days, everything feels pointless and I don't see the future")
predict_depression("I've stopped talking to everyone, I just don't see the point anymore, nothing brings me joy")
predict_depression("I keep thinking everyone would be better off without me around")

# Group 2 — Should clearly be NORMAL (low probability)
predict_depression("Busy week at college but feeling good, went out with friends on Friday")
predict_depression("Stressed about exams but confident I'll get through it, just need to focus")
predict_depression("Feeling a bit tired today but overall life is going well, excited for the holidays")

# Group 3 — Tricky borderline cases (tests model nuance)
predict_depression("I smile at work every day but when I get home I just sit in the dark and feel nothing")
predict_depression("I've been meditating and journaling lately, taking time to process my emotions")
predict_depression("Lost my job last week, feeling worried about money but trying to stay positive")

# Group 4 — False positive traps (should be NORMAL but might trigger model)
predict_depression("I prefer being alone, always been an introvert, love my quiet evenings")
predict_depression("Took a break from social media, feels so much healthier and peaceful")
predict_depression("Cried during a movie today, it was just really emotional and beautiful")

# Group 5 — Subtle depression (easy to miss, tests recall)
predict_depression("I don't know, I just feel kind of grey lately, not sad exactly, just nothing")
predict_depression("Everyone keeps saying I seem fine but I genuinely cannot remember the last time I felt happy")
predict_depression("I keep making plans and then cancelling them, I just can't explain why I can't show up")

Text: I haven't left my bed in days, everything feels pointless and I don't see the fu...
Depression probability: 0.7483
Normal probability:     0.2517
Prediction: DEPRESSION
------------------------------------------------------------
Text: I've stopped talking to everyone, I just don't see the point anymore, nothing br...
Depression probability: 0.8935
Normal probability:     0.1065
Prediction: DEPRESSION
------------------------------------------------------------
Text: I keep thinking everyone would be better off without me around...
Depression probability: 0.8966
Normal probability:     0.1034
Prediction: DEPRESSION
------------------------------------------------------------
Text: Busy week at college but feeling good, went out with friends on Friday...
Depression probability: 0.0010
Normal probability:     0.9990
Prediction: NORMAL
------------------------------------------------------------
Text: Stressed about exams but confident I'll get through it, just need to focus...
Depr